In [7]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

class IQRClipper(BaseEstimator, TransformerMixin):
    """
    Кастомный трансформер для ограничения выбросов по методу IQR.
    Статистики (Q1, Q3, границы) вычисляются строго на обучающей выборке в fit().
    """
    def __init__(self, factor=1.5):
        self.factor = factor
        self.lower_bounds_ = {}
        self.upper_bounds_ = {}
        
    def fit(self, X, y=None):
        X_df = pd.DataFrame(X)
        for col in X_df.columns:
            Q1 = X_df[col].quantile(0.25)
            Q3 = X_df[col].quantile(0.75)
            IQR = Q3 - Q1
            self.lower_bounds_[col] = Q1 - self.factor * IQR
            self.upper_bounds_[col] = Q3 + self.factor * IQR
        return self
        
    def transform(self, X):
        X_df = pd.DataFrame(X).copy()
        for col in X_df.columns:
            if col in self.lower_bounds_:
                X_df[col] = np.clip(
                    X_df[col], 
                    self.lower_bounds_[col], 
                    self.upper_bounds_[col]
                )
        return X_df.values

In [8]:
from sklearn.model_selection import train_test_split

# Загружаем наш датасет из первой работы
df = pd.read_csv('../data/raw/churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].str.strip(), errors='coerce')

# Выделяем признаки (X) и целевую переменную (y)
X = df.drop(columns=['customerID', 'Churn'])
y = df['Churn'].apply(lambda x: 1 if x == 'Yes' else 0)

# ВАЖНО: Сначала делим выборку, ДО любых заполнений пропусков и вычислений статистик!
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Размер train: {X_train.shape}")
print(f"Размер test: {X_test.shape}")

Размер train: (5634, 19)
Размер test: (1409, 19)


In [9]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

# Выбираем числовые и категориальные признаки для обработки
numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_features = ['gender', 'Contract', 'PaymentMethod']

# Пайплайн для числовых признаков: заполняем пропуски медианой -> убираем выбросы (наш кастомный класс) -> масштабируем
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('clipper', IQRClipper(factor=1.5)),
    ('scaler', StandardScaler())
])

# Пайплайн для категориальных признаков: заполняем пропуски константой -> One-Hot кодирование
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Объединяем оба подпайплайна через ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Финальный пайплайн: Предобработка + Модель (RandomForest)
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

# ВАЖНО: Обучаем ВСЮ систему СТРОГО на обучающей выборке
pipeline.fit(X_train, y_train)

# Проверяем точность на тестовой выборке
accuracy = pipeline.score(X_test, y_test)
print(f"Accuracy пайплайна на тесте: {accuracy:.4f}")

Accuracy пайплайна на тесте: 0.7821


In [10]:
def test_clipper_fit_transform():
    # Подготовка учебных данных с явными выбросами
    train_data = pd.DataFrame({'feature': [10, 20, 25, 30, 1000]})
    test_data = pd.DataFrame({'feature': [15, 2000]})  # 2000 — сильный выброс в тесте
    
    clipper = IQRClipper(factor=1.5)
    clipper.fit(train_data)
    
    # Тест 1: Проверяем, что границы успешно зафиксировались по train
    assert 'feature' in clipper.lower_bounds_
    assert 'feature' in clipper.upper_bounds_
    
    # Тест 2: Применяем к новым данным (выброс 2000 должен обрезаться по верхнему порогу train)
    transformed_test = clipper.transform(test_data)
    
    # Убеждаемся, что значение 2000 было ограничено (оно стало меньше 2000)
    assert transformed_test[1, 0] < 2000
    # Проверяем отсутствие пропусков/NaN на выходе
    assert not np.isnan(transformed_test).any()
    
    print("✅ Все тесты трансформера успешно пройдены!")

# Запускаем написанный тест
test_clipper_fit_transform()

✅ Все тесты трансформера успешно пройдены!


In [11]:
import time

# Замер времени обучения пайплайна на train-выборке
start_time = time.time()
pipeline.fit(X_train, y_train)
fit_time = time.time() - start_time

# Замер времени предсказания на test-выборке
start_time = time.time()
_ = pipeline.predict(X_test)
predict_time = time.time() - start_time

print(f"⏱️ Время обучения пайплайна: {fit_time:.4f} секунд")
print(f"⏱️ Время предсказания: {predict_time:.4f} секунд")

⏱️ Время обучения пайплайна: 0.4613 секунд
⏱️ Время предсказания: 0.0289 секунд


## Анализ производительности и узкие места (Bottlenecks)
1. **Конвертация типов данных:** Внутри кастомного трансформера `IQRClipper` выполняется преобразование матриц в `pd.DataFrame`, что создает дополнительные накладные расходы по памяти и процессорному времени при работе с миллионными датасетами.
2. **One-Hot Encoding:** При высокой кардинальности категориальных признаков (множество уникальных значений) размерность закодированной матрицы резко возрастает, что замедляет работу моделей и увеличивает потребление RAM.